In [ ]:
from dataset import Dataset
from model import Retriever, Augmenter, Generator, RetrievalEvaluator
from evaluate import Evaluator
import os

import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

In [ ]:
note_prompts = {
    "easy": "Important Note: Your output will strictly be Yes or No with no other words.",
    "medium": "Important Note: Your output must be strictly formatted as a comma-separated list of nutrients prefixed with “high” or “low”, based solely on the provided options: carb, protein, sugar, sodium, cholesterol, saturated_fat, calorie. \
        For example, a valid output would be: high_carb, low_protein, high_sugar. No extra words or deviations are allowed.",
    "hard": "Important Note: Your output must consist of “Yes” or “No”, followed by a list of nutrients addressed with “high” or “low,” selected from the following options: carb, protein, sugar, sodium, cholesterol, saturated fat, and calorie. \
        For example, a valid output would be: Yes, because the food is high in carb, low in protein, high in sugar. Ensure the output adheres to this format without any additional words or deviations.",
}

method_prompts = {
    "plain": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "KAPING": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "ToG": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "zero_cot": "Let's think step by step.",
    "cot_bag": "Let's construct a graph with the nodes and edges first."
}

In [ ]:

# Modify the parameters here to find good prompts.
api_key = os.getenv("API_KEY")
file_path = "../processed_data/NGQA_benchmark.csv"

task_level = "medium"
question_level = "medium"
is_sample = True
n = 50
model_name = "gpt-4o-mini"
method = "ToG"

note_prompt = note_prompts.get(task_level)
method_prompt = method_prompts.get(method)

data = Dataset(file_path)
questions, answers, graphs = data.process(question_level=question_level, task_level=task_level, sample=is_sample, n=n)

retriever = Retriever(graphs, model_name=model_name)
retrieved_graphs = retriever.retrieve(method=method, api_key=api_key, questions=questions)

retrieval_evaluator = RetrievalEvaluator(graphs)
retrieval_evaluation_results = retrieval_evaluator.evaluate(retrieved_graphs)
print('Retrieval evaluation results:', retrieval_evaluation_results)

augmenter = Augmenter()
textualized_graphs = augmenter.augment(retrieved_graphs)

generator = Generator(api_key = api_key, 
                      model_name=model_name, note_prompt=note_prompt, method_prompt=method_prompt)
predictions = generator.generate_predictions(questions, textualized_graphs)

evaluator = Evaluator()
final_output_results = evaluator.evaluate(task_level, predictions, answers)
print('Final output evaluation results:', final_output_results)